# 갑상선암 검진 정책 변화 분석 (학생용 Colab, 수정판)

이 노트북은 **GitHub → Colab에서 열어 `런타임 > 모두 실행`** 하면 분석이 진행되도록 만든 **수정판**입니다.

수정 사항 핵심:
- 외부 스크립트 의존을 제거하고, **로컬에 고정된 분석 스크립트**를 생성해 사용
- **연령미상/범주형 NaN** 때문에 중간에 멈추던 오류 수정
- **ITS 변수 정의**를 표준적인 형태로 보정
- CSV 로딩, 요약문 생성, 저장 단계의 **예외 내성 강화**


In [ ]:
import os, sys, subprocess, urllib.request, shutil, warnings
from urllib.parse import quote
from pathlib import Path

warnings.filterwarnings("ignore")

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

print("환경 설정 중...")

if in_colab():
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "matplotlib", "statsmodels", "openpyxl", "scipy"],
    check=True
)

print("환경 설정 완료")

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False
print("라이브러리 로드 완료")

In [ ]:
# =========================
# 저장소 고정 설정
# =========================
GITHUB_USER = "runmc3812"
REPO_NAME = "Healthcare_Informatics_Integration-2026-"
BRANCH = "main"
BASE_PATH = "1조"

DATA_DIR = "data"
MAIN_FILE = "국립암센터_24개종 암발생률_20260120.csv"
AGE_FILE = "24개_암종_성_연령_5세_별_암발생자수__발생률_20260324142549.csv"

OUTDIR = "/content/thyroid_analysis_outputs"
os.makedirs(OUTDIR, exist_ok=True)

def gh_raw_url(*parts):
    encoded = "/".join(quote(str(p), safe="") for p in parts)
    return f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{encoded}"

main_source = gh_raw_url(BASE_PATH, DATA_DIR, MAIN_FILE)
age_source = gh_raw_url(BASE_PATH, DATA_DIR, AGE_FILE)

print("main_source =", main_source)
print("age_source  =", age_source)

In [ ]:
# 원본 데이터 다운로드
local_main = "/content/main_thyroid_24cancers.csv"
local_age = "/content/age_thyroid_by_5y.csv"

def safe_download(url, dest):
    try:
        urllib.request.urlretrieve(url, dest)
        print("다운로드 완료:", dest)
    except Exception as e:
        raise RuntimeError(f"다운로드 실패: {url}\n{e}")

safe_download(main_source, local_main)
safe_download(age_source, local_age)

In [ ]:
# 수정된 분석 스크립트를 로컬에 저장 후 import
fixed_script_path = "/content/thyroid_colab_analysis_fixed.py"

FIXED_SCRIPT = r'''
from __future__ import annotations
import os
import re
import math
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple
from urllib.parse import quote
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import statsmodels.api as sm

POLICY_YEAR = 2014
PEAK_YEAR = 2012
COMPARE_YEARS = [2012, 2015, 2023]


def setup_korean_font() -> None:
    font_candidates = [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    ]
    for font_path in font_candidates:
        if os.path.exists(font_path):
            try:
                fm.fontManager.addfont(font_path)
                mpl.rcParams["font.family"] = fm.FontProperties(fname=font_path).get_name()
                break
            except Exception:
                pass
    mpl.rcParams["axes.unicode_minus"] = False
    mpl.rcParams["figure.dpi"] = 160


def gh_raw_url(user: str, repo: str, branch: str, *parts: str) -> str:
    encoded = "/".join(quote(str(p), safe="") for p in parts)
    return f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/{encoded}"


def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path


def download_file(url: str, dest: str) -> str:
    ensure_dir(str(Path(dest).parent))
    urlretrieve(url, dest)
    return dest


def clean_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    if s in {"", "-", "nan", "None"}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def safe_pct_change(a, b):
    if pd.isna(a) or a == 0 or pd.isna(b):
        return np.nan
    return (b - a) / a * 100


def age_midpoint(age_str):
    s = str(age_str).strip()
    if s in {"", "계", "연령미상", "nan"}:
        return np.nan
    m = re.match(r"(\d+)-(\d+)세", s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return (a + b) / 2
    if "85세이상" in s:
        return 87.5
    return np.nan


def read_csv_flex(path_or_url: str, **kwargs) -> pd.DataFrame:
    last_error = None
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            return pd.read_csv(path_or_url, encoding=enc, low_memory=False, **kwargs)
        except Exception as e:
            last_error = e
    raise last_error


def load_main_incidence(source: str) -> pd.DataFrame:
    df = read_csv_flex(source)
    required = {"발생연도", "성별", "암종", "발생자수", "조발생률", "연령표준화발생률"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"주요 발생률 파일의 필수 열이 없습니다: {sorted(missing)}")
    for c in ["발생자수", "조발생률", "연령표준화발생률"]:
        df[c] = df[c].apply(clean_num)
    df["성별"] = df["성별"].astype(str).str.strip()
    df["암종"] = df["암종"].astype(str).str.strip()
    df = df[df["암종"].str.contains("갑상선", na=False)].copy()
    df = df.sort_values(["성별", "발생연도"]).reset_index(drop=True)
    return df


def load_age_sex_incidence(source: str) -> pd.DataFrame:
    raw = read_csv_flex(source, header=None)
    if raw.shape[0] < 3 or raw.shape[1] < 5:
        raise ValueError("연령별 파일 형식이 예상과 다릅니다.")

    year_row = raw.iloc[0].astype(str).str.strip().tolist()
    metric_row = raw.iloc[1].astype(str).str.strip().tolist()

    new_cols = []
    for i, (y, m) in enumerate(zip(year_row, metric_row)):
        if i == 0:
            new_cols.append("암종")
        elif i == 1:
            new_cols.append("성별")
        elif i == 2:
            new_cols.append("연령군")
        else:
            year = re.search(r"(\d{4})", y)
            if not year:
                raise ValueError(f"연도 헤더를 해석할 수 없습니다: {y}")
            metric = "발생자수" if "발생자수" in m else "조발생률"
            new_cols.append(f"{year.group(1)}_{metric}")

    df = raw.iloc[2:].copy()
    df.columns = new_cols
    df["암종"] = df["암종"].astype(str).str.strip()
    df["성별"] = df["성별"].astype(str).str.strip()
    df["연령군"] = df["연령군"].astype(str).str.strip()

    df = df[df["암종"].str.contains("갑상선", na=False)].copy()

    id_vars = ["암종", "성별", "연령군"]
    value_vars = [c for c in df.columns if c not in id_vars]

    long_df = df.melt(
        id_vars=id_vars,
        value_vars=value_vars,
        var_name="year_metric",
        value_name="value",
    )
    long_df["연도"] = long_df["year_metric"].str.extract(r"(\d{4})").astype(int)
    long_df["지표"] = long_df["year_metric"].str.extract(r"_(발생자수|조발생률)")
    long_df["value"] = long_df["value"].apply(clean_num)

    long_df = (
        long_df.pivot_table(
            index=["암종", "성별", "연령군", "연도"],
            columns="지표",
            values="value",
            aggfunc="first",
            observed=False,
        )
        .reset_index()
    )
    long_df.columns.name = None

    long_df = long_df[~long_df["연령군"].isin(["0-4세", "연령미상"])].copy()
    long_df["연령중앙값"] = long_df["연령군"].apply(age_midpoint)

    age_order = [
        "계",
        "5-9세", "10-14세", "15-19세", "20-24세", "25-29세",
        "30-34세", "35-39세", "40-44세", "45-49세", "50-54세",
        "55-59세", "60-64세", "65-69세", "70-74세", "75-79세",
        "80-84세", "85세이상",
    ]
    long_df["연령군"] = pd.Categorical(long_df["연령군"], categories=age_order, ordered=True)
    long_df = long_df.sort_values(["성별", "연령군", "연도"]).reset_index(drop=True)
    return long_df


def save_df(df: pd.DataFrame, outdir: str, name: str) -> Tuple[str, str]:
    ensure_dir(outdir)
    csv_path = os.path.join(outdir, f"{name}.csv")
    xlsx_path = os.path.join(outdir, f"{name}.xlsx")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_excel(xlsx_path, index=False)
    return csv_path, xlsx_path


def savefig(outdir: str, name: str) -> str:
    ensure_dir(outdir)
    path = os.path.join(outdir, name)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    return path


def draw_policy_line(policy_year: int = POLICY_YEAR) -> None:
    plt.axvline(policy_year, linestyle="--", linewidth=1.5)


def make_index(df: pd.DataFrame, year_col: str, value_col: str, base_year: int) -> pd.DataFrame:
    tmp = df.copy()
    base_series = tmp.loc[tmp[year_col] == base_year, value_col]
    if base_series.empty or pd.isna(base_series.iloc[0]) or base_series.iloc[0] == 0:
        raise ValueError(f"기준연도 {base_year}의 값이 없어 지수를 계산할 수 없습니다.")
    base = base_series.iloc[0]
    tmp[f"index_{base_year}=100"] = tmp[value_col] / base * 100
    return tmp


def age_change_table(df: pd.DataFrame, sex_label: str) -> pd.DataFrame:
    pivot = df.pivot_table(
        index="연령군",
        columns="연도",
        values="조발생률",
        aggfunc="first",
        observed=False,
    )
    out = pd.DataFrame(index=pivot.index)
    out["2012"] = pivot.get(2012)
    out["2015"] = pivot.get(2015)
    out["2023"] = pivot.get(2023)
    out["2012→2015 변화율(%)"] = [safe_pct_change(a, b) for a, b in zip(out["2012"], out["2015"])]
    out["2015→2023 변화율(%)"] = [safe_pct_change(a, b) for a, b in zip(out["2015"], out["2023"])]
    out["성별"] = sex_label
    return out.reset_index()


def weighted_mean_age(df: pd.DataFrame) -> pd.DataFrame:
    tmp = df[df["연령군"] != "계"].copy()
    tmp = tmp.dropna(subset=["연령중앙값", "발생자수"])
    out = (
        tmp.groupby("연도", group_keys=False)
        .apply(lambda x: np.average(x["연령중앙값"], weights=x["발생자수"]))
        .reset_index(name="가중평균연령")
    )
    return out


def fit_its(df: pd.DataFrame, year_col: str, value_col: str, policy_year: int = POLICY_YEAR):
    d = df[[year_col, value_col]].dropna().copy().sort_values(year_col)
    d["time"] = d[year_col] - d[year_col].min()
    d["post"] = (d[year_col] >= policy_year).astype(int)
    d["time_after_policy"] = (d[year_col] - policy_year).clip(lower=0)

    X = sm.add_constant(d[["time", "post", "time_after_policy"]], has_constant="add")
    y = d[value_col]
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 1})
    d["fitted"] = model.predict(X)

    pre = d[d[year_col] < policy_year].copy()
    if len(pre) >= 3:
        X_pre = sm.add_constant(pre[["time"]], has_constant="add")
        pre_model = sm.OLS(pre[value_col], X_pre).fit()
        X_cf = sm.add_constant(d[["time"]], has_constant="add")
        d["counterfactual"] = pre_model.predict(X_cf)
        d.loc[d[year_col] < policy_year, "counterfactual"] = np.nan
    else:
        d["counterfactual"] = np.nan

    coef = pd.DataFrame(
        {
            "term": model.params.index,
            "coef": model.params.values,
            "pvalue": model.pvalues.values,
            "ci_low": model.conf_int()[0].values,
            "ci_high": model.conf_int()[1].values,
        }
    )
    return model, d, coef


def its_table_and_plot(df_main: pd.DataFrame, intervention_year: int, out_png: str):
    d = (
        df_main[(df_main["암종"] == "갑상선") & (df_main["성별"] == "남녀전체")]
        .copy()
        .sort_values("발생연도")[["발생연도", "연령표준화발생률"]]
        .rename(columns={"연령표준화발생률": "ASR"})
    )
    model, fit_df, coef = fit_its(d, "발생연도", "ASR", policy_year=intervention_year)
    plt.figure(figsize=(10, 5))
    plt.plot(fit_df["발생연도"], fit_df["ASR"], marker="o", label="Observed ASR")
    plt.plot(fit_df["발생연도"], fit_df["fitted"], linestyle="--", label=f"ITS fitted ({intervention_year})")
    if fit_df["counterfactual"].notna().any():
        plt.plot(fit_df["발생연도"], fit_df["counterfactual"], linestyle=":", label="Counterfactual")
    plt.axvline(intervention_year, linestyle=":", linewidth=2, label=f"Intervention {intervention_year}")
    plt.title(f"갑상선암 ITS 분석 ({intervention_year} 권고안 기준)")
    plt.xlabel("연도")
    plt.ylabel("연령표준화발생률")
    plt.legend()
    ensure_dir(str(Path(out_png).parent))
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close()

    label_map = {
        "const": "절편",
        "time": "시간추세",
        "post": "개입직후 수준변화",
        "time_after_policy": "개입후 기울기변화",
    }
    coef_df = coef.copy()
    coef_df["변수"] = coef_df["term"].map(label_map).fillna(coef_df["term"])
    coef_df = coef_df[["변수", "coef", "pvalue", "ci_low", "ci_high"]].rename(
        columns={"coef": "계수", "pvalue": "p값", "ci_low": "CI하한", "ci_high": "CI상한"}
    )
    return fit_df, coef_df


def sex_age_rebound(long_df: pd.DataFrame) -> pd.DataFrame:
    df = long_df[
        (long_df["성별"].isin(["남자", "여자"]))
        & (long_df["연령군"] != "계")
    ].copy()
    wide = df.pivot_table(index=["성별", "연령군"], columns="연도", values="조발생률", aggfunc="first", observed=False)
    for y in [2015, 2023]:
        if y not in wide.columns:
            wide[y] = np.nan
    out = wide[[2015, 2023]].copy()
    out.columns = ["2015 조발생률", "2023 조발생률"]
    out["2015→2023 변화율(%)"] = [safe_pct_change(a, b) for a, b in zip(out["2015 조발생률"], out["2023 조발생률"])]
    out = out.replace([np.inf, -np.inf], np.nan).reset_index()
    return out


def rebound_contribution(long_df: pd.DataFrame) -> pd.DataFrame:
    df = long_df[(long_df["성별"] == "계") & (long_df["연령군"] != "계")].copy()
    wide = df.pivot_table(index="연령군", columns="연도", values="발생자수", aggfunc="first", observed=False)
    for y in [2015, 2023]:
        if y not in wide.columns:
            wide[y] = np.nan
    out = wide[[2015, 2023]].copy()
    out.columns = ["2015 발생자수", "2023 발생자수"]
    out["증가분"] = out["2023 발생자수"] - out["2015 발생자수"]
    pos_sum = out.loc[out["증가분"] > 0, "증가분"].sum()
    out["기여도(%)"] = np.where((out["증가분"] > 0) & (pos_sum > 0), out["증가분"] / pos_sum * 100, np.nan)
    out = out.sort_values("증가분", ascending=False).reset_index()
    return out


def cancer_benchmark(df_main: pd.DataFrame) -> pd.DataFrame:
    df = df_main[df_main["성별"] == "남녀전체"].copy()
    wide = df.pivot_table(index="암종", columns="발생연도", values="연령표준화발생률", aggfunc="first")
    for y in [2012, 2015, 2023]:
        if y not in wide.columns:
            wide[y] = np.nan
    out = wide[[2012, 2015, 2023]].copy()
    out.columns = ["2012 ASR", "2015 ASR", "2023 ASR"]
    out["2012→2015 변화율(%)"] = [safe_pct_change(a, b) for a, b in zip(out["2012 ASR"], out["2015 ASR"])]
    out["2015→2023 변화율(%)"] = [safe_pct_change(a, b) for a, b in zip(out["2015 ASR"], out["2023 ASR"])]
    out = out.replace([np.inf, -np.inf], np.nan).reset_index()
    return out


def build_additional_summary(sex_age_df: pd.DataFrame, contrib_df: pd.DataFrame, benchmark_df: pd.DataFrame) -> str:
    lines = ["[추가 요약]"]
    lines.append("- 2014년 권고안 기준 ITS를 별도로 생성하여 정책 분기점을 직접 점검했습니다.")

    top_rebound = sex_age_df.dropna(subset=["2015→2023 변화율(%)"]).sort_values("2015→2023 변화율(%)", ascending=False).head(5)
    if not top_rebound.empty:
        top_labels = [f"{r['성별']} {r['연령군']} ({r['2015→2023 변화율(%)']:.1f}%)" for _, r in top_rebound.iterrows()]
        lines.append("- 성별×연령 rebound 상위군: " + ", ".join(top_labels))
    else:
        lines.append("- 성별×연령 rebound 상위군을 계산할 충분한 데이터가 없었습니다.")

    top_contrib = contrib_df.dropna(subset=["증가분"]).sort_values("증가분", ascending=False).head(5)
    if not top_contrib.empty:
        contrib_labels = [f"{r['연령군']} (+{int(r['증가분'])}명)" for _, r in top_contrib.iterrows() if pd.notna(r["증가분"])]
        lines.append("- 전체 rebound 기여 상위 연령군: " + ", ".join(contrib_labels))

    thyroid_rank = benchmark_df.dropna(subset=["2015→2023 변화율(%)"]).sort_values("2015→2023 변화율(%)", ascending=False).reset_index(drop=True)
    thyroid_idx = thyroid_rank.index[thyroid_rank["암종"] == "갑상선"].tolist()
    if thyroid_idx:
        lines.append(f"- 24개 암종 비교에서 갑상선암의 2015→2023 rebound 순위는 {thyroid_idx[0] + 1}위입니다.")
    return "\n".join(lines)


def run_analysis(
    main_source: str,
    age_source: str,
    outdir: str = "thyroid_analysis_outputs",
) -> Dict[str, pd.DataFrame]:
    setup_korean_font()
    ensure_dir(outdir)

    main_df = load_main_incidence(main_source)
    age_df = load_age_sex_incidence(age_source)

    main_df.to_csv(os.path.join(outdir, "thyroid_incidence_main.csv"), index=False, encoding="utf-8-sig")
    age_df.to_csv(os.path.join(outdir, "thyroid_age_sex_long.csv"), index=False, encoding="utf-8-sig")

    overall = main_df[main_df["성별"] == "남녀전체"].copy()
    male = main_df[main_df["성별"] == "남자"].copy()
    female = main_df[main_df["성별"] == "여자"].copy()

    age_total = age_df[age_df["성별"] == "계"].copy()
    age_male = age_df[age_df["성별"] == "남자"].copy()
    age_female = age_df[age_df["성별"] == "여자"].copy()

    age_total_only = age_total[age_total["연령군"] != "계"].copy()
    age_male_only = age_male[age_male["연령군"] != "계"].copy()
    age_female_only = age_female[age_female["연령군"] != "계"].copy()

    def get_asr(df, year):
        s = df.loc[df["발생연도"] == year, "연령표준화발생률"]
        return s.iloc[0] if len(s) else np.nan

    summary_change = pd.DataFrame(
        [
            {
                "집단": label,
                "2012_ASR": get_asr(d, 2012),
                "2015_ASR": get_asr(d, 2015),
                "2023_ASR": get_asr(d, 2023),
                "2012→2015 변화율(%)": safe_pct_change(get_asr(d, 2012), get_asr(d, 2015)),
                "2015→2023 변화율(%)": safe_pct_change(get_asr(d, 2015), get_asr(d, 2023)),
                "2012→2023 변화율(%)": safe_pct_change(get_asr(d, 2012), get_asr(d, 2023)),
            }
            for label, d in [("전체", overall), ("남자", male), ("여자", female)]
        ]
    )
    summary_change.to_csv(os.path.join(outdir, "summary_change_overall_sex.csv"), index=False, encoding="utf-8-sig")

    age_change_total = age_change_table(age_total_only, "계")
    age_change_male = age_change_table(age_male_only, "남자")
    age_change_female = age_change_table(age_female_only, "여자")
    age_change_all = pd.concat([age_change_total, age_change_male, age_change_female], ignore_index=True)
    age_change_all.to_csv(os.path.join(outdir, "age_change_all.csv"), index=False, encoding="utf-8-sig")

    # 1 overall trend
    plt.figure(figsize=(10, 5))
    plt.plot(overall["발생연도"], overall["연령표준화발생률"], marker="o", linewidth=2.5)
    draw_policy_line()
    plt.title("갑상선암 연령표준화발생률 추세 (전체)")
    plt.xlabel("연도")
    plt.ylabel("연령표준화발생률")
    savefig(outdir, "figure01_overall_asr_trend.png")

    # 2 sex trend
    plt.figure(figsize=(10, 5))
    plt.plot(male["발생연도"], male["연령표준화발생률"], marker="o", linewidth=2, label="남자")
    plt.plot(female["발생연도"], female["연령표준화발생률"], marker="o", linewidth=2, label="여자")
    draw_policy_line()
    plt.title("갑상선암 연령표준화발생률 추세 (남녀 비교)")
    plt.xlabel("연도")
    plt.ylabel("연령표준화발생률")
    plt.legend()
    savefig(outdir, "figure02_sex_asr_trend.png")

    # 3 shock index
    overall_idx = make_index(overall, "발생연도", "연령표준화발생률", PEAK_YEAR)
    male_idx = make_index(male, "발생연도", "연령표준화발생률", PEAK_YEAR)
    female_idx = make_index(female, "발생연도", "연령표준화발생률", PEAK_YEAR)
    female_mid = age_female_only[age_female_only["연령군"].isin(["50-54세", "55-59세"])].copy()
    female_mid = female_mid.groupby("연도", as_index=False)["조발생률"].mean()
    base_mid = female_mid.loc[female_mid["연도"] == PEAK_YEAR, "조발생률"].iloc[0]
    female_mid[f"index_{PEAK_YEAR}=100"] = female_mid["조발생률"] / base_mid * 100

    plt.figure(figsize=(10, 5))
    plt.plot(overall_idx["발생연도"], overall_idx[f"index_{PEAK_YEAR}=100"], marker="o", linewidth=2, label="전체")
    plt.plot(male_idx["발생연도"], male_idx[f"index_{PEAK_YEAR}=100"], marker="o", linewidth=2, label="남자")
    plt.plot(female_idx["발생연도"], female_idx[f"index_{PEAK_YEAR}=100"], marker="o", linewidth=2, label="여자")
    plt.plot(female_mid["연도"], female_mid[f"index_{PEAK_YEAR}=100"], marker="o", linewidth=2, label="여성 50–59세(평균)")
    draw_policy_line()
    plt.title(f"갑상선암 발생 충격지수 ({PEAK_YEAR}=100)")
    plt.xlabel("연도")
    plt.ylabel("지수")
    plt.legend()
    savefig(outdir, "figure03_shock_index_2012_100.png")

    # 4 sex ratio
    sex_ratio = overall[["발생연도"]].copy()
    sex_ratio["여자_ASR"] = female["연령표준화발생률"].values
    sex_ratio["남자_ASR"] = male["연령표준화발생률"].values
    sex_ratio["여성/남성 비율"] = sex_ratio["여자_ASR"] / sex_ratio["남자_ASR"]
    sex_ratio.to_csv(os.path.join(outdir, "sex_ratio_timeseries.csv"), index=False, encoding="utf-8-sig")

    plt.figure(figsize=(10, 5))
    plt.plot(sex_ratio["발생연도"], sex_ratio["여성/남성 비율"], marker="o", linewidth=2.2)
    draw_policy_line()
    plt.title("갑상선암 여성/남성 발생률 비율 추세")
    plt.xlabel("연도")
    plt.ylabel("여성/남성 비율")
    savefig(outdir, "figure04_female_male_ratio.png")

    # 5 age profiles
    plot_df = age_total_only[age_total_only["연도"].isin(COMPARE_YEARS)].copy()
    plot_df = plot_df.dropna(subset=["연령군", "조발생률"])
    order = [a for a in plot_df["연령군"].cat.categories if a in plot_df["연령군"].astype(str).unique()]
    plt.figure(figsize=(10, 5))
    for yr in COMPARE_YEARS:
        g = plot_df[plot_df["연도"] == yr].copy().sort_values("연령중앙값")
        g = g.dropna(subset=["연령군", "조발생률"])
        plt.plot(g["연령군"].astype(str), g["조발생률"], marker="o", linewidth=2, label=str(yr))
    plt.xticks(rotation=45)
    plt.title("갑상선암 연령군별 조발생률 프로파일 (전체)")
    plt.xlabel("연령군")
    plt.ylabel("조발생률")
    plt.legend()
    savefig(outdir, "figure05_age_profile_total_2012_2015_2023.png")

    # 6 butterfly
    years = [2012, 2015, 2023]
    fig, axes = plt.subplots(1, 3, figsize=(16, 8), sharey=True)
    for ax, yr in zip(axes, years):
        m = age_male_only[age_male_only["연도"] == yr].copy().sort_values("연령중앙값")
        f = age_female_only[age_female_only["연도"] == yr].copy().sort_values("연령중앙값")
        m = m.dropna(subset=["연령군", "조발생률"])
        f = f.dropna(subset=["연령군", "조발생률"])
        y = np.arange(len(m))
        labels = m["연령군"].astype(str).tolist()
        ax.barh(y, -m["조발생률"].values, alpha=0.8, label="남자")
        ax.barh(y, f["조발생률"].values, alpha=0.8, label="여자")
        ax.axvline(0, linewidth=1)
        ax.set_title(f"{yr}")
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.set_xlabel("조발생률")
    axes[0].legend(loc="lower right")
    fig.suptitle("갑상선암 발생의 성별·연령 구조 변화 (버터플라이 비교)")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(outdir, "figure06_butterfly_2012_2015_2023.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # 7/8 heatmaps
    heat_order = age_total_only[["연령군", "연령중앙값"]].dropna(subset=["연령군"]).drop_duplicates().sort_values("연령중앙값")["연령군"]
    heat = age_total_only.dropna(subset=["연령군"]).pivot(index="연령군", columns="연도", values="조발생률").loc[heat_order]
    plt.figure(figsize=(12, 7))
    plt.imshow(heat.values, aspect="auto")
    plt.colorbar(label="조발생률")
    plt.xticks(np.arange(len(heat.columns)), heat.columns, rotation=90)
    plt.yticks(np.arange(len(heat.index)), [str(x) for x in heat.index])
    plt.title("갑상선암 연령군별 조발생률 Heatmap (전체)")
    plt.xlabel("연도")
    plt.ylabel("연령군")
    savefig(outdir, "figure07_heatmap_total.png")

    heat_f = age_female_only.dropna(subset=["연령군"]).pivot(index="연령군", columns="연도", values="조발생률").loc[heat_order]
    plt.figure(figsize=(12, 7))
    plt.imshow(heat_f.values, aspect="auto")
    plt.colorbar(label="조발생률")
    plt.xticks(np.arange(len(heat_f.columns)), heat_f.columns, rotation=90)
    plt.yticks(np.arange(len(heat_f.index)), [str(x) for x in heat_f.index])
    plt.title("갑상선암 연령군별 조발생률 Heatmap (여성)")
    plt.xlabel("연도")
    plt.ylabel("연령군")
    savefig(outdir, "figure08_heatmap_female.png")

    # 9/10/11 waterfall
    wf1 = age_change_total[age_change_total["연령군"] != "계"].copy().sort_values("연령군")
    plt.figure(figsize=(11, 6))
    plt.bar(wf1["연령군"].astype(str), wf1["2012→2015 변화율(%)"])
    plt.xticks(rotation=45)
    plt.title("연령군별 감소율: 2012 → 2015 (전체)")
    plt.xlabel("연령군")
    plt.ylabel("변화율 (%)")
    savefig(outdir, "figure09_waterfall_drop_total_2012_2015.png")

    wf2 = age_change_total[age_change_total["연령군"] != "계"].copy().sort_values("연령군")
    plt.figure(figsize=(11, 6))
    plt.bar(wf2["연령군"].astype(str), wf2["2015→2023 변화율(%)"])
    plt.xticks(rotation=45)
    plt.title("연령군별 rebound: 2015 → 2023 (전체)")
    plt.xlabel("연령군")
    plt.ylabel("변화율 (%)")
    savefig(outdir, "figure10_waterfall_rebound_total_2015_2023.png")

    wf3 = age_change_female[age_change_female["연령군"] != "계"].copy().sort_values("연령군")
    plt.figure(figsize=(11, 6))
    plt.bar(wf3["연령군"].astype(str), wf3["2012→2015 변화율(%)"])
    plt.xticks(rotation=45)
    plt.title("연령군별 감소율: 2012 → 2015 (여성)")
    plt.xlabel("연령군")
    plt.ylabel("변화율 (%)")
    savefig(outdir, "figure11_waterfall_drop_female_2012_2015.png")

    # 12 bubble
    bubble = age_df[(age_df["연도"] == 2023) & (age_df["연령군"] != "계")].copy()
    bubble = bubble.dropna(subset=["연령중앙값", "조발생률", "발생자수"])
    bubble["size"] = bubble["발생자수"] / bubble["발생자수"].max() * 1800
    plt.figure(figsize=(11, 6))
    for sex in ["남자", "여자"]:
        g = bubble[bubble["성별"] == sex].copy()
        plt.scatter(g["연령중앙값"], g["조발생률"], s=g["size"], alpha=0.6, label=sex)
    xtick_df = bubble.drop_duplicates("연령중앙값").sort_values("연령중앙값")
    plt.xticks(xtick_df["연령중앙값"], xtick_df["연령군"].astype(str), rotation=45)
    plt.title("갑상선암 성별·연령별 burden (2023)")
    plt.xlabel("연령군")
    plt.ylabel("조발생률")
    plt.legend()
    savefig(outdir, "figure12_bubble_2023.png")

    # 13 weighted mean age
    mean_age_total = weighted_mean_age(age_total)
    mean_age_male = weighted_mean_age(age_male)
    mean_age_female = weighted_mean_age(age_female)
    plt.figure(figsize=(10, 5))
    plt.plot(mean_age_total["연도"], mean_age_total["가중평균연령"], marker="o", linewidth=2, label="전체")
    plt.plot(mean_age_male["연도"], mean_age_male["가중평균연령"], marker="o", linewidth=2, label="남자")
    plt.plot(mean_age_female["연도"], mean_age_female["가중평균연령"], marker="o", linewidth=2, label="여자")
    draw_policy_line()
    plt.title("갑상선암 발생의 가중평균연령 추세 (근사치)")
    plt.xlabel("연도")
    plt.ylabel("연령(세)")
    plt.legend()
    savefig(outdir, "figure13_weighted_mean_age.png")

    mean_age_all = mean_age_total.merge(mean_age_male, on="연도", suffixes=("_전체", "_남자"))
    mean_age_all = mean_age_all.merge(mean_age_female, on="연도")
    mean_age_all = mean_age_all.rename(columns={"가중평균연령": "가중평균연령_여자"})
    mean_age_all.to_csv(os.path.join(outdir, "weighted_mean_age.csv"), index=False, encoding="utf-8-sig")

    # 14 ratio by age group
    sel_ages = ["20-24세", "30-34세", "40-44세", "50-54세", "60-64세"]
    ratio_rows = []
    for ag in sel_ages:
        m = age_male_only[age_male_only["연령군"] == ag][["연도", "조발생률"]].rename(columns={"조발생률": "남자"})
        f = age_female_only[age_female_only["연령군"] == ag][["연도", "조발생률"]].rename(columns={"조발생률": "여자"})
        r = pd.merge(m, f, on="연도", how="inner")
        if r.empty:
            continue
        r["연령군"] = ag
        r["여성/남성 비율"] = np.where(r["남자"] != 0, r["여자"] / r["남자"], np.nan)
        ratio_rows.append(r)
    if ratio_rows:
        ratio_age = pd.concat(ratio_rows, ignore_index=True)
        ratio_age.to_csv(os.path.join(outdir, "ratio_by_agegroup.csv"), index=False, encoding="utf-8-sig")

        plt.figure(figsize=(11, 6))
        for ag in sel_ages:
            g = ratio_age[ratio_age["연령군"] == ag]
            if not g.empty:
                plt.plot(g["연도"], g["여성/남성 비율"], marker="o", linewidth=2, label=ag)
        draw_policy_line()
        plt.title("연령군별 여성/남성 비율 추세")
        plt.xlabel("연도")
        plt.ylabel("여성/남성 비율")
        plt.legend(ncol=2)
        savefig(outdir, "figure14_ratio_by_agegroup.png")

    # ITS
    its_results = {}
    for label, df_ in [("전체", overall), ("남자", male), ("여자", female)]:
        model, fit_df, coef = fit_its(df_, "발생연도", "연령표준화발생률", policy_year=POLICY_YEAR)
        its_results[label] = {"model": model, "fit_df": fit_df, "coef": coef}
        coef.to_csv(os.path.join(outdir, f"its_coef_{label}.csv"), index=False, encoding="utf-8-sig")
        fit_df.to_csv(os.path.join(outdir, f"its_fit_{label}.csv"), index=False, encoding="utf-8-sig")

        plt.figure(figsize=(10, 5))
        plt.plot(fit_df["발생연도"], fit_df["연령표준화발생률"], marker="o", linewidth=2, label="관측값")
        plt.plot(fit_df["발생연도"], fit_df["fitted"], linewidth=2, label="ITS 적합선")
        if fit_df["counterfactual"].notna().any():
            plt.plot(fit_df["발생연도"], fit_df["counterfactual"], linestyle="--", linewidth=2, label="예상 기대선")
        draw_policy_line()
        plt.title(f"ITS: 갑상선암 연령표준화발생률 ({label})")
        plt.xlabel("연도")
        plt.ylabel("연령표준화발생률")
        plt.legend()
        savefig(outdir, f"figure_its_{label}.png")

    # summary
    y1999 = get_asr(overall, 1999)
    y2012 = get_asr(overall, 2012)
    y2015 = get_asr(overall, 2015)
    y2023 = get_asr(overall, 2023)

    lines = [
        "===== 자동 해석 =====",
        f"1) 전체 연령표준화발생률은 1999년 {y1999:.1f}에서 2012년 {y2012:.1f}로 증가했다.",
        f"2) 이후 2015년 {y2015:.1f}로 감소하여, 2012년 대비 {safe_pct_change(y2012, y2015):.1f}% 변화하였다.",
        f"3) 2023년 {y2023:.1f}로 일부 재상승했으나, 2012년 정점에는 미치지 못했다.",
        "",
        "4) 2012→2015 감소폭 상위 연령군",
        age_change_total.sort_values("2012→2015 변화율(%)").head(5).to_string(index=False),
        "",
        "5) 2015→2023 rebound 상위 연령군",
        age_change_total.sort_values("2015→2023 변화율(%)", ascending=False).head(5).to_string(index=False),
    ]

    for label in ["전체", "남자", "여자"]:
        coef = its_results[label]["coef"].copy()
        post = coef.loc[coef["term"] == "post", "coef"].iloc[0]
        post_p = coef.loc[coef["term"] == "post", "pvalue"].iloc[0]
        slope = coef.loc[coef["term"] == "time_after_policy", "coef"].iloc[0]
        slope_p = coef.loc[coef["term"] == "time_after_policy", "pvalue"].iloc[0]
        lines += [
            "",
            f"[{label}]",
            f"- 정책 직후 수준 변화(post): {post:.3f} (p={post_p:.4f})",
            f"- 정책 후 추세 변화(time_after_policy): {slope:.3f} (p={slope_p:.4f})",
        ]

    with open(os.path.join(outdir, "auto_summary.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return {
        "main_df": main_df,
        "age_df": age_df,
        "summary_change": summary_change,
        "age_change_total": age_change_total,
        "age_change_female": age_change_female,
        "outdir": outdir,
    }
'''

with open(fixed_script_path, "w", encoding="utf-8") as f:
    f.write(FIXED_SCRIPT)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from thyroid_colab_analysis_fixed import (
    run_analysis,
    read_csv_flex,
    load_main_incidence,
    load_age_sex_incidence,
    save_df,
    its_table_and_plot,
    sex_age_rebound,
    rebound_contribution,
    cancer_benchmark,
    build_additional_summary,
)

print("수정된 분석 스크립트 로드 완료")

## 1. 기존 분석 실행

In [ ]:
print("\n기존 분석 실행 중...")
results = run_analysis(main_source=local_main, age_source=local_age, outdir=OUTDIR)

display(Markdown("## 핵심 요약표"))
display(results["summary_change"])

display(Markdown("## 2012→2015 감소폭 상위 10개 연령군"))
display(results["age_change_total"].sort_values("2012→2015 변화율(%)").head(10))

display(Markdown("## 2015→2023 rebound 상위 10개 연령군"))
display(results["age_change_total"].sort_values("2015→2023 변화율(%)", ascending=False).head(10))

## 2. 추가 분석용 데이터 로드

In [ ]:
main_df = read_csv_flex(local_main)
age_long = load_age_sex_incidence(local_age)

print("main_df shape =", main_df.shape)
print("age_long shape =", age_long.shape)
display(age_long.head())

## 3. 2014 권고안 기준 ITS

In [ ]:
its_series_2014, its_coef_2014 = its_table_and_plot(
    load_main_incidence(local_main),
    intervention_year=2014,
    out_png=os.path.join(OUTDIR, "its_2014_policy.png")
)
save_df(its_coef_2014, OUTDIR, "its_2014_coefficients")

display(Markdown("## 추가분석 1: 2014 권고안 기준 ITS 계수"))
display(its_coef_2014)
display(Markdown(f"- 그림 저장: `{os.path.join(OUTDIR, 'its_2014_policy.png')}`"))

## 4. 성별 × 연령별 rebound

In [ ]:
sex_age_df = sex_age_rebound(age_long)
save_df(sex_age_df, OUTDIR, "sex_age_rebound")

top10 = sex_age_df.sort_values("2015→2023 변화율(%)", ascending=False).head(10)
display(Markdown("## 추가분석 2: 성별×연령별 rebound 상위 10개"))
display(top10)

top_plot = sex_age_df.dropna(subset=["2015→2023 변화율(%)"]).sort_values("2015→2023 변화율(%)", ascending=False).head(12)
labels = top_plot["성별"].astype(str) + " / " + top_plot["연령군"].astype(str)
plt.figure(figsize=(10, 6))
plt.barh(labels.iloc[::-1], top_plot["2015→2023 변화율(%)"].iloc[::-1])
plt.title("성별 × 연령별 갑상선암 Rebound 상위군 (2015→2023)")
plt.xlabel("변화율(%)")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "sex_age_rebound_top12.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close()

## 5. 전체 rebound 기여 연령군

In [ ]:
contrib_df = rebound_contribution(age_long)
save_df(contrib_df, OUTDIR, "rebound_contribution")

display(Markdown("## 추가분석 3: 전체 rebound 기여 연령군 상위 10개"))
display(contrib_df.head(10))

top = contrib_df.dropna(subset=["증가분"]).head(12)
plt.figure(figsize=(10, 6))
plt.barh(top["연령군"].astype(str).iloc[::-1], top["증가분"].iloc[::-1])
plt.title("2015→2023 전체 Rebound 기여 연령군 (증가분 기준)")
plt.xlabel("발생자수 증가분")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "rebound_contribution_top12.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close()

## 6. 24개 암종 비교

In [ ]:
benchmark_df = cancer_benchmark(main_df)
save_df(benchmark_df, OUTDIR, "cancer_benchmark_24")

display(Markdown("## 추가분석 4: 24개 암종 중 rebound 상위 10개"))
display(benchmark_df.sort_values("2015→2023 변화율(%)", ascending=False).head(10))

rebound_top = benchmark_df.dropna(subset=["2015→2023 변화율(%)"]).sort_values("2015→2023 변화율(%)", ascending=False).head(10)
plt.figure(figsize=(10, 6))
plt.barh(rebound_top["암종"].astype(str).iloc[::-1], rebound_top["2015→2023 변화율(%)"].iloc[::-1])
plt.title("24개 암종 중 2015→2023 Rebound 상위 10개")
plt.xlabel("변화율(%)")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "cancer_rebound_top10.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close()

## 7. 자동 요약 생성

In [ ]:
summary_path = os.path.join(OUTDIR, "auto_summary.txt")
extra_summary_path = os.path.join(OUTDIR, "auto_summary_additional.txt")

extra_text = build_additional_summary(sex_age_df, contrib_df, benchmark_df)
with open(extra_summary_path, "w", encoding="utf-8") as f:
    f.write(extra_text)

if os.path.exists(summary_path):
    with open(summary_path, "r", encoding="utf-8") as f:
        print("\n[기존 자동 요약]\n")
        print(f.read())

if os.path.exists(extra_summary_path):
    with open(extra_summary_path, "r", encoding="utf-8") as f:
        print("\n" + f.read())

## 8. 결과 압축 및 다운로드

In [ ]:
zip_path = shutil.make_archive("/content/thyroid_analysis_outputs", "zip", OUTDIR)
print("\n압축 완료:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("자동 다운로드는 Colab에서만 동작합니다:", e)

print("\n완료: GitHub의 이 노트북을 Colab에서 열어 `모두 실행`하면 전체 분석이 생성됩니다.")